In [ ]:
# Lab type: review
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Index Maintenance: Freshness, Updates, and Embedding Drift
# Task: The document-update handler below runs a live demonstration of
# its own defect. Run it, watch the orphan appear, and answer the
# judgment questions.

# Lab: Reviewing an Index Update Path

We use FAISS with an ID-mapped index so vectors can be added and removed by chunk ID — the same mechanics as a production vector database, small enough to inspect.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

In [ ]:
import faiss

DIM = DOC_EMB.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(DIM))

# chunk registry: numeric id -> (doc_id, chunk_no, text)
registry = {}
next_id = 0

def add_chunks(doc_id, chunk_texts):
    global next_id
    vecs = embed(chunk_texts)
    ids = np.arange(next_id, next_id + len(chunk_texts))
    index.add_with_ids(np.asarray(vecs, dtype="float32"), ids)
    for i, (cid, text) in enumerate(zip(ids, chunk_texts)):
        registry[int(cid)] = (doc_id, i, text)
    next_id += len(chunk_texts)

# Index the refund policy as 3 chunks (v1 of the document)
V1_CHUNKS = [
    "Billing > Refunds: full refund within 30 days of purchase.",
    "Billing > Refunds: after 30 days contact billing support.",
    "Billing > Refunds: annual subscriptions get prorated partial refunds.",
]
add_chunks("refund-policy", V1_CHUNKS)
print(f"index size: {index.ntotal}, registry: {len(registry)}")

## The update handler under review

In [ ]:
# --- THE UPDATE HANDLER (review this code — is it correct?) ---
def update_document(doc_id, new_chunk_texts):
    """Re-chunk and upsert a changed document."""
    vecs = embed(new_chunk_texts)
    # overwrite chunk i of this doc with new chunk i
    ids = [cid for cid, (d, i, t) in sorted(registry.items())
           if d == doc_id][:len(new_chunk_texts)]
    index.remove_ids(np.array(ids, dtype="int64"))
    index.add_with_ids(np.asarray(vecs, dtype="float32"),
                       np.array(ids, dtype="int64"))
    for cid, text in zip(ids, new_chunk_texts):
        registry[cid] = (doc_id, registry[cid][1], text)

# v2 of the policy: the 30-day window became 14 days, and the policy is
# now SHORTER — it re-chunks to 2 chunks, not 3.
V2_CHUNKS = [
    "Billing > Refunds: full refund within 14 days of purchase.",
    "Billing > Refunds: no partial refunds for annual subscriptions.",
]
update_document("refund-policy", V2_CHUNKS)
print(f"index size after update: {index.ntotal}")

def search_chunks(query, k=3):
    scores, ids = index.search(
        np.asarray(embed([query]), dtype="float32"), k)
    return [(registry[int(i)][2], float(s))
            for s, i in zip(scores[0], ids[0]) if i != -1]

for text, score in search_chunks("do annual subscriptions get partial refunds"):
    print(f"  {score:.3f}  {text}")

**Question 1.** The search above asked about partial refunds for annual subscriptions. v2 of the policy says there are none — yet look at what ranked highly. Which chunk is it, which document version does it belong to, and exactly which line(s) of `update_document` let it survive?

<details>
<summary>🔑 Reveal answer — Question 1</summary>

The v1 chunk "annual subscriptions get prorated partial refunds" is still in the index — the exact *opposite* of current policy. `update_document` slices the doc's chunk IDs to `[:len(new_chunk_texts)]` (2 of the 3) and replaces only those; v1's third chunk is never touched. This is the orphan defect: chunk-count changes on re-publish leave the old tail retrievable forever, and both policy versions now answer queries. The correct shape is **delete-everything-for-doc_id first** (`remove_ids` over *all* the doc's chunks, then add the new ones under fresh IDs).

</details>

**Question 2.** Write the corrected `update_document_fixed` (delete-then-insert by document), apply it (re-run the v2 update), and show the stale chunk is gone from the same search.

In [ ]:
# Work here: implement update_document_fixed(doc_id, new_chunk_texts)
# then re-run the search from above.


<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def update_document_fixed(doc_id, new_chunk_texts):
    stale = [cid for cid, (d, i, t) in registry.items() if d == doc_id]
    index.remove_ids(np.array(stale, dtype="int64"))
    for cid in stale:
        del registry[cid]
    add_chunks(doc_id, new_chunk_texts)

update_document_fixed("refund-policy", V2_CHUNKS)
print(f"index size: {index.ntotal}")   # 2 — v1 fully gone
for text, score in search_chunks("do annual subscriptions get partial refunds"):
    print(f"  {score:.3f}  {text}")
```

In production this delete-then-insert should also be atomic (or hidden behind a version field) so queries mid-update don't see a half-updated document.

</details>

**Question 3.** Nothing in this notebook pins which embedding model built the index. Describe the failure sequence if the query side upgrades to a different embedding model while these stored vectors remain, and the one-line assertion from the lesson that converts it into a loud error.

<details>
<summary>🔑 Reveal answer — Question 3</summary>

Query vectors from model v2 are compared against document vectors from v1 — different embedding spaces — so similarity scores become noise and recall decays diffusely with no exception anywhere (dimensions often match, so nothing crashes). The guard: store `embedding_model` in the index metadata and assert at query time that the query-side model matches it — turning a silent quality collapse into a deployment error. A model upgrade then means a scheduled full-corpus re-embed.

</details>

**Question 4.** This lab's evals would not have caught the stale-chunk defect if the eval set had been labelled before the policy changed. Name the index-level monitors from the lesson that catch freshness failures answer-level evals cannot see.

<details>
<summary>🔑 Reveal answer — Question 4</summary>

Ingestion lag (newest indexed change vs newest corpus change), per-document index age, orphan and tombstone counts, and — highest signal — a canary set of recently *edited* documents whose new content is verified retrievable (and old content verified gone) after each ingestion cycle.

</details>

## Summary

1. Updates must be delete-then-insert keyed by _______, because re-chunking changes chunk counts.
2. An orphaned chunk keeps _______ forever, with no error.
3. An embedding model upgrade invalidates _______ stored vector.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **document (doc_id)**
2. **answering queries / being retrievable**
3. **every**

</details>